Generate Datasets


In [6]:
from huggingface_hub import login
from dotenv import load_dotenv
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TextStreamer,
    BitsAndBytesConfig,
)
import torch
import os

print("Imports successful!")

Imports successful!


In [7]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


In [12]:
load_dotenv(override=True)
hf_token = os.getenv('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [13]:
LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"

In [8]:
system_message = """
You generate realistic synthetic customer support data for a tattoo website.
The website allows customers to explore tattoo designs, ask questions about
tattoos, learn about aftercare, compare artist portfolios, and book appointments.

Generate diverse, realistic customer questions and helpful responses in a
structured format. Include questions about booking, pricing, tattoo designs,
pain, preparation, aftercare, artist availability, cancellations, and rescheduling.

Do not invent actual artist names, real bookings, or confirmed prices.
Use fictional examples and clearly mark prices or policies as illustrative.
"""

user_prompt = """
Generate 10 synthetic customer support examples for a tattoo website.

Include a variety of topics:
- Booking a tattoo appointment
- Tattoo pricing and deposits
- Choosing tattoo designs and sizes
- Questions about tattoo pain
- Tattoo aftercare
- Artist availability
- Cancellations and rescheduling

For each example, provide:
- customer_question
- assistant_response
- category
- difficulty (easy, medium, or hard)

Make the examples diverse, realistic, and non-repetitive.
Return the results as valid JSON only.
"""


messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
]



In [10]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
streamer = TextStreamer(tokenizer)
model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)
outputs = model.generate(inputs, max_new_tokens=2000, streamer=streamer)